In [98]:
import numpy as np
import pandas as pd
from pygmid import Lookup as lk

#### Technology data

In [99]:
n = lk('../gf180mcuD/simulation/nfet_03v3.mat')
p = lk('../gf180mcuD/simulation/pfet_03v3.mat')

#### Specifications

In [100]:
c_load = 50e-15
gm_id_m12 = 10
gm_id_m34 = 5
gm_id_m56 = 5
l_12 = 0.3
l_34 = 0.3
l_56 = 0.6
f_bw = 500e6
i_bias_in = 100e-6
i_total_limit = 200e-6


In [101]:
gm_m12 = f_bw * 3 * 4*np.pi*c_load
print('gm12 =', round(gm_m12/1e-3, 4), 'mS')

gm12 = 0.9425 mS


In [102]:
id_m12 = gm_m12 / gm_id_m12
i_total = 2*id_m12
print('i_total (exact) =', round(i_total/1e-6, 1), 'µA')

i_total = max(round(i_total / 1e-6 * 2) / 2 * 1e-6, 0.5e-6)
id_m12 = i_total/2

print('i_total (rounded) =', i_total/1e-6, 'µA')
if i_total < i_total_limit:
    print('[info] power consumption target is met!')
else:
    print('[info] power consumption target is NOT met!') 

i_total (exact) = 188.5 µA
i_total (rounded) = 188.5 µA
[info] power consumption target is met!


In [103]:
gm_gds_m12 = n.lookup('GM_GDS', GM_ID=gm_id_m12, L=l_12, VDS=1.65, VSB=0)
gm_gds_m34 = n.lookup('GM_GDS', GM_ID=gm_id_m34, L=l_34, VDS=1.65, VSB=0)

gds_m12 = gm_m12 / gm_gds_m12
gm_m34 = gm_id_m34 * i_total/2
gds_m34 = gm_m34 / gm_gds_m34

a0 = gm_m12 / (gds_m12 + gds_m34)
print('a0 =', round(20*np.log10(a0), 1), 'dB')

a0 = 27.7 dB


In [104]:
gm_cgs_m12 = n.lookup('GM_CGS', GM_ID=gm_id_m12, L=l_12, VDS=1.65, VSB=0)
gm_cdd_m12 = n.lookup('GM_CDD', GM_ID=gm_id_m12, L=l_12, VDS=1.65, VSB=0)
gm_cdd_m34 = p.lookup('GM_CDD', GM_ID=gm_id_m34, L=l_34, VDS=1.65, VSB=0)

c_load_parasitic = abs(gm_m12/gm_cgs_m12) + abs(gm_m12/gm_cdd_m12) + abs(gm_m34/gm_cdd_m34)
print('additional load capacitance =', round(c_load_parasitic/1e-15, 1), 'fF')

f_bw = gm_m12 / (4*np.pi * (c_load + c_load_parasitic))
print('unity gain bandwidth incl. parasitics =', round(f_bw/1e6, 2), 'MHz')

additional load capacitance = 31.9 fF
unity gain bandwidth incl. parasitics = 915.67 MHz


In [105]:
vgs_m12 = n.look_upVGS(GM_ID=gm_id_m12, L=l_12, VDS=1.65, VSB=0.0)
vgs_m34 = p.look_upVGS(GM_ID=gm_id_m34, L=l_34, VDS=1.65, VSB=0.0) 
vgs_m56 = n.look_upVGS(GM_ID=gm_id_m56, L=l_56, VDS=1.65, VSB=0.0) 

print('vgs_12 =', round(float(vgs_m12), 3), 'V')
print('vgs_34 =', round(float(vgs_m34), 3), 'V')
print('vgs_56 =', round(float(vgs_m56), 3), 'V')

vgs_12 = 0.756 V
vgs_34 = 1.079 V
vgs_56 = 1.037 V


In [106]:
# calculate all widths
id_w_m12 = n.lookup('ID_W', GM_ID=gm_id_m12, L=l_12, VDS=vgs_m12, VSB=0)
w_12 = id_m12 / id_w_m12
w_12_round = max(round(w_12*2)/2, 0.5)
print('M1/2 W =', round(w_12, 2), 'um, rounded W =', w_12_round, 'um')

id_m34 = id_m12
id_w_m34 = p.lookup('ID_W', GM_ID=gm_id_m34, L=l_34, VDS=vgs_m34, VSB=0)
w_34 = id_m34 / id_w_m34
w_34_round = max(round(w_34*2)/2, 0.5) 
print('M3/4 W =', round(w_34, 2), 'um, rounded W =', w_34_round, 'um')

id_w_m5 = n.lookup('ID_W', GM_ID=gm_id_m56, L=l_56, VDS=vgs_m56, VSB=0)
w_5 = i_total / id_w_m5
w_5_round = max(round(w_5*2)/2, 0.5)
print('M5 W =', round(w_5, 2), 'um, rounded W =', w_5_round, 'um')
w_6 = w_5_round * i_bias_in / i_total
w_6_round = max(round(w_6*2)/2, 0.5)
print('M6 W =', round(w_6_round, 2), 'um')

M1/2 W = 14.86 um, rounded W = 15.0 um
M3/4 W = 10.81 um, rounded W = 11.0 um
M5 W = 12.91 um, rounded W = 13.0 um
M6 W = 7.0 um
